In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CSV_PATH = 'hf://datasets/hafizhrafizal/suno-discord-chat-history/Suno - SUNO HUB - 💬┃general-chat [1069381916492562585].csv'

df = pd.read_csv(CSV_PATH, low_memory=False)
df['Date'] = pd.to_datetime(df['Date'], format='mixed', utc=True)
df = df.sort_values('Date').reset_index(drop=True)

df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.to_period('M')
df['Week']       = df['Date'].dt.to_period('W').dt.start_time
df['DayOfWeek']  = df['Date'].dt.day_name()
df['Hour']       = df['Date'].dt.hour
df['MsgLength']  = df['Content'].astype(str).str.len()

df.head()


C:\Users\Hafizh Rafizal Adnan\AppData\Local\Temp\ipykernel_28460\3612061639.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['Month']      = df['Date'].dt.to_period('M')
C:\Users\Hafizh Rafizal Adnan\AppData\Local\Temp\ipykernel_28460\3612061639.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['Week']       = df['Date'].dt.to_period('W').dt.start_time


,AuthorID,Author,Date,Content,Attachments,Reactions,Year,Month,Week,DayOfWeek,Hour,MsgLength
0,687471333403131955,kmfrey,2023-01-29 22:24:02.654000+00:00,Joined the server.,NaN,NaN,2023,2023-01,2023-01-23,Sunday,22,18
1,1069379683088605284,suno_ai_,2023-01-29 22:24:40.979000+00:00,Welcome!,NaN,NaN,2023,2023-01,2023-01-23,Sunday,22,8
2,1069853697221337138,sunomc,2023-03-07 21:36:33.859000+00:00,Joined the server.,NaN,NaN,2023,2023-03,2023-03-06,Tuesday,21,18
3,159985870458322944,MEE6#4876,2023-03-07 22:47:04.261000+00:00,Joined the server.,NaN,NaN,2023,2023-03,2023-03-06,Tuesday,22,18
4,1033432389516546158,veryboldbagel,2023-04-06 18:40:25.008000+00:00,👋,NaN,NaN,2023,2023-04,2023-04-03,Thursday,18,1


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaleido'], check=True)

from pathlib import Path

VIZ_DIR = Path('data_visualization')
VIZ_DIR.mkdir(exist_ok=True)

def save_fig(fig, name):
    path = str(VIZ_DIR / f'{name}.png')
    fig.write_image(path, scale=2, width=1200, height=fig.layout.height or 500)
    print(f'  Saved: {path}')


## 1. Dataset Overview

In [2]:
print("=== Dataset Overview ===")
print(f"Total messages : {len(df):,}")
print(f"Unique authors : {df['Author'].nunique():,}")
print(f"Date range     : {df['Date'].min().date()} → {df['Date'].max().date()}")

print("\nMessages per year:")
print(df.groupby('Year').size().rename('Messages').to_string())

print("\nNull counts (key columns):")
print(df[['Content', 'Attachments', 'Reactions']].isnull().sum().to_string())

print("\nColumn dtypes:")
print(df.dtypes.to_string())


=== Dataset Overview ===
Total messages : 3,324,452
Unique authors : 40,127
Date range     : 2023-01-29 → 2025-12-31

Messages per year:
Year
2023      54986
2024    1403899
2025    1865567

Null counts (key columns):
Content           2529
Attachments    3324447
Reactions      3318351

Column dtypes:
AuthorID                     int64
Author                      object
Date           datetime64[ns, UTC]
Content                     object
Attachments                 object
Reactions                   object
Year                         int32
Month                    period[M]
Week                datetime64[ns]
DayOfWeek                   object
Hour                         int32
MsgLength                    int64


## 2. Message Volume Trends

In [ ]:
# Monthly message volume
monthly = df.groupby('Month').size().reset_index(name='Messages')
monthly['Month_str'] = monthly['Month'].astype(str)

fig = px.bar(monthly, x='Month_str', y='Messages',
             title='Monthly Message Volume — Suno Discord',
             labels={'Month_str': 'Month', 'Messages': 'Messages'})
fig.update_layout(xaxis_tickangle=-45, height=500, hovermode='x unified')
fig.show()
save_fig(fig, '01_monthly_volume')


In [ ]:
# Weekly message volume
weekly = df.groupby('Week').size().reset_index(name='Messages')

fig = px.line(weekly, x='Week', y='Messages',
              title='Weekly Message Volume — Suno Discord',
              labels={'Week': 'Week', 'Messages': 'Messages'},
              markers=True)
fig.update_layout(hovermode='x unified', height=500)
fig.show()
save_fig(fig, '02_weekly_volume')


In [ ]:
# Activity heatmap: Day of Week × Hour (UTC)
DOW_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = df.groupby(['DayOfWeek', 'Hour']).size().reset_index(name='Messages')
pivot = (heatmap_data
         .pivot(index='DayOfWeek', columns='Hour', values='Messages')
         .reindex(DOW_ORDER))

fig = px.imshow(pivot,
                title='Message Activity Heatmap — Day of Week × Hour (UTC)',
                labels={'x': 'Hour (UTC)', 'y': 'Day of Week', 'color': 'Messages'},
                color_continuous_scale='Blues',
                aspect='auto')
fig.update_layout(height=400)
fig.show()
save_fig(fig, '03_heatmap_dow_hour')


## 3. Author Activity

In [ ]:
# Top 30 authors by total message count
top_authors = (df.groupby('Author').size()
               .sort_values(ascending=False)
               .head(30)
               .reset_index(name='Messages'))

fig = px.bar(top_authors, x='Messages', y='Author', orientation='h',
             title='Top 30 Authors by Total Messages',
             labels={'Messages': 'Total Messages', 'Author': 'Author'},
             color='Messages', color_continuous_scale='Reds')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=700)
fig.show()
save_fig(fig, '04_top30_authors')


In [ ]:
# Author activity index — monthly message heatmap for top 30 authors
top30_names = (df.groupby('Author').size()
               .sort_values(ascending=False)
               .head(30).index.tolist())

df_top30 = df[df['Author'].isin(top30_names)].copy()
df_top30['Month_str'] = df_top30['Month'].astype(str)

activity = df_top30.groupby(['Author', 'Month_str']).size().reset_index(name='Messages')
pivot_activity = (activity
                  .pivot(index='Author', columns='Month_str', values='Messages')
                  .fillna(0))
pivot_activity = pivot_activity.loc[top30_names]

fig = px.imshow(pivot_activity,
                title='Author Activity Index — Monthly Messages (Top 30)',
                labels={'x': 'Month', 'y': 'Author', 'color': 'Messages'},
                color_continuous_scale='YlOrRd',
                aspect='auto')
fig.update_layout(height=800, xaxis_tickangle=-45)
fig.show()
save_fig(fig, '05_author_activity_index')


In [ ]:
# New users per month — first message appearance
first_month = df.groupby('Author')['Month'].min().reset_index()
first_month.columns = ['Author', 'FirstMonth']
new_users = first_month.groupby('FirstMonth').size().reset_index(name='NewUsers')
new_users['FirstMonth_str'] = new_users['FirstMonth'].astype(str)

fig = px.bar(new_users, x='FirstMonth_str', y='NewUsers',
             title='New Users per Month (First Message Appearance)',
             labels={'FirstMonth_str': 'Month', 'NewUsers': 'New Users'})
fig.update_layout(xaxis_tickangle=-45, height=500, hovermode='x unified')
fig.show()
save_fig(fig, '06_new_users_per_month')


In [ ]:
# Author longevity — number of unique active months (top 30 by longevity)
longevity = (df.groupby('Author')['Month'].nunique()
             .sort_values(ascending=False)
             .head(30)
             .reset_index(name='ActiveMonths'))

fig = px.bar(longevity, x='ActiveMonths', y='Author', orientation='h',
             title='Author Longevity — Unique Active Months (Top 30)',
             labels={'ActiveMonths': 'Months Active', 'Author': 'Author'},
             color='ActiveMonths', color_continuous_scale='Teal')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=700)
fig.show()
save_fig(fig, '07_author_longevity')


## 4. Suno Admin Activity

In [10]:
SUNO_ADMIN = [
    "kmfrey", "nedi_suno", "sunomc", "gkucsko", "mikeysuno",
    "david_at_suno", "theothermikefromsuno", "suno_ai_",
    "hellinahandbasket7", "alisaadeddine_03226", "bigtony2651",
    "patsuno", "suno_z", "sunoap", "united_rabbit_70663_77768", ".hunos"
]

df['isSunoTeam'] = df['Author'].isin(SUNO_ADMIN)
df_admin = df[df['isSunoTeam']].copy()

print(f"Admin messages    : {len(df_admin):,}")
print(f"Community messages: {len(df[~df['isSunoTeam']]):,}")
print(f"Admin share       : {len(df_admin)/len(df)*100:.2f}%")


Admin messages    : 7,779
Community messages: 3,316,673
Admin share       : 0.23%


In [ ]:
# Admin vs Community message volume over time (stacked bar)
monthly_split = (df.groupby(['Month', 'IsAdmin']).size()
                 .unstack(fill_value=0)
                 .rename(columns={False: 'Community', True: 'Admin'})
                 .reset_index())
monthly_split['Month_str'] = monthly_split['Month'].astype(str)

fig = go.Figure()
fig.add_bar(x=monthly_split['Month_str'], y=monthly_split['Community'], name='Community', marker_color='steelblue')
fig.add_bar(x=monthly_split['Month_str'], y=monthly_split['Admin'],     name='Suno Admin',  marker_color='tomato')
fig.update_layout(barmode='stack',
                  title='Monthly Messages — Suno Admin vs Community',
                  xaxis_tickangle=-45, height=500, hovermode='x unified')
fig.show()
save_fig(fig, '08_admin_vs_community')


In [ ]:
# Individual admin message count
admin_counts = (df_admin.groupby('Author').size()
                .sort_values(ascending=False)
                .reset_index(name='Messages'))

fig = px.bar(admin_counts, x='Author', y='Messages',
             title='Individual Suno Admin — Total Message Count',
             labels={'Messages': 'Total Messages', 'Author': 'Admin'},
             color='Messages', color_continuous_scale='Oranges')
fig.update_layout(height=450)
fig.show()
save_fig(fig, '09_individual_admin_count')


In [ ]:
# Admin activity heatmap — each admin's monthly message count
df_admin['Month_str'] = df_admin['Month'].astype(str)
admin_monthly = df_admin.groupby(['Author', 'Month_str']).size().reset_index(name='Messages')
pivot_admin = (admin_monthly
               .pivot(index='Author', columns='Month_str', values='Messages')
               .fillna(0))

fig = px.imshow(pivot_admin,
                title='Suno Admin Activity Index — Monthly Messages',
                labels={'x': 'Month', 'y': 'Admin', 'color': 'Messages'},
                color_continuous_scale='Greens',
                aspect='auto')
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()
save_fig(fig, '10_admin_activity_heatmap')


In [15]:
# Save admin messages to CSV
df_admin.drop(columns=['IsAdmin', 'Year', 'Month', 'Week', 'DayOfWeek', 'Hour', 'MsgLength', 'Month_str'], errors='ignore').to_csv('Suno Discord Admin Messages.csv', index=False)
print("Saved: Suno Discord Admin Messages.csv")


Saved: Suno Discord Admin Messages.csv


## 5. Focused Analysis: April 2023 – June 2024

In [ ]:
# Filter to Apr 2023 – Jun 2024
PERIOD_START = pd.Period('2023-04', freq='M')
PERIOD_END   = pd.Period('2024-06', freq='M')

df_p = df[(df['Month'] >= PERIOD_START) & (df['Month'] <= PERIOD_END)].copy()
df_p['Month_str'] = df_p['Month'].astype(str)

print(f"Messages in period : {len(df_p):,}")
print(f"Unique authors     : {df_p['Author'].nunique():,}")
print(f"Date range         : {df_p['Date'].min().date()} → {df_p['Date'].max().date()}")


### 5.1 Weekly Volume with Key Event Annotations

In [ ]:
# Weekly volume with annotated key events
KEY_EVENTS = {
    '2023-07-24': 'Suno lawsuit filed',
    '2024-02-21': 'Suno v3 Alpha released',
    '2024-03-18': 'Suno v3 public release',
    '2024-04-02': 'Artist Rights Alliance letter',
    '2024-05-01': 'Sony Music warning',
    '2024-05-21': 'Series B funding / v3.5 launch',
}

weekly_p = df_p.groupby('Week').size().reset_index(name='Messages')

fig = px.line(weekly_p, x='Week', y='Messages', markers=True,
              title='Weekly Message Volume — Apr 2023 to Jun 2024',
              labels={'Week': 'Week', 'Messages': 'Messages'})

for date_str, label in KEY_EVENTS.items():
    fig.add_vline(x=date_str, line_dash='dash', line_color='red', line_width=1.2)
    fig.add_annotation(x=date_str, y=1, yref='paper',
                       text=label, textangle=-90,
                       showarrow=False, yanchor='bottom',
                       font=dict(size=10, color='red'))

fig.update_layout(hovermode='x unified', height=550)
fig.show()
save_fig(fig, '11_period_weekly_events')


### 5.2 Monthly Active Authors & Messages per Author

In [ ]:
# Unique active authors per month vs total messages (dual axis)
monthly_stats = df_p.groupby('Month_str').agg(
    Messages=('Author', 'count'),
    UniqueAuthors=('Author', 'nunique')
).reset_index()
monthly_stats['MsgPerAuthor'] = (monthly_stats['Messages'] / monthly_stats['UniqueAuthors']).round(1)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=monthly_stats['Month_str'], y=monthly_stats['Messages'],
                     name='Total Messages', marker_color='steelblue'), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly_stats['Month_str'], y=monthly_stats['UniqueAuthors'],
                         name='Unique Authors', mode='lines+markers',
                         line=dict(color='tomato', width=2)), secondary_y=True)
fig.update_layout(title='Monthly Messages vs Unique Active Authors — Apr 2023 to Jun 2024',
                  xaxis_tickangle=-45, height=500, hovermode='x unified')
fig.update_yaxes(title_text='Total Messages', secondary_y=False)
fig.update_yaxes(title_text='Unique Active Authors', secondary_y=True)
fig.show()
save_fig(fig, '12_period_messages_vs_authors')

fig2 = px.line(monthly_stats, x='Month_str', y='MsgPerAuthor', markers=True,
               title='Average Messages per Active Author per Month',
               labels={'Month_str': 'Month', 'MsgPerAuthor': 'Msgs / Author'})
fig2.update_layout(height=400, hovermode='x unified')
fig2.show()
save_fig(fig2, '12b_period_msg_per_author')


### 5.3 Top Authors in Period & Activity Index

In [ ]:
# Top 30 authors in the period
top_p = (df_p.groupby('Author').size()
         .sort_values(ascending=False)
         .head(30)
         .reset_index(name='Messages'))

fig = px.bar(top_p, x='Messages', y='Author', orientation='h',
             title='Top 30 Authors by Messages — Apr 2023 to Jun 2024',
             labels={'Messages': 'Total Messages', 'Author': 'Author'},
             color='Messages', color_continuous_scale='Reds')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=700)
fig.show()
save_fig(fig, '13_period_top30_authors')


In [ ]:
# Author activity index heatmap for period (top 30 in this period)
top30_p = top_p['Author'].tolist()
activity_p = (df_p[df_p['Author'].isin(top30_p)]
              .groupby(['Author', 'Month_str']).size()
              .reset_index(name='Messages'))
pivot_p = (activity_p
           .pivot(index='Author', columns='Month_str', values='Messages')
           .fillna(0)
           .loc[top30_p])

fig = px.imshow(pivot_p,
                title='Author Activity Index — Monthly Messages, Apr 2023 to Jun 2024 (Top 30)',
                labels={'x': 'Month', 'y': 'Author', 'color': 'Messages'},
                color_continuous_scale='YlOrRd',
                aspect='auto')
fig.update_layout(height=800, xaxis_tickangle=-45)
fig.show()
save_fig(fig, '14_period_activity_index')


### 5.4 New Users & Author Retention

In [ ]:
# New users per month within the period
first_p = (df_p.groupby('Author')['Month'].min()
           .reset_index()
           .rename(columns={'Month': 'FirstMonth'}))
new_p = first_p.groupby('FirstMonth').size().reset_index(name='NewUsers')
new_p['FirstMonth_str'] = new_p['FirstMonth'].astype(str)

fig = px.bar(new_p, x='FirstMonth_str', y='NewUsers',
             title='New Users per Month — Apr 2023 to Jun 2024',
             labels={'FirstMonth_str': 'Month', 'NewUsers': 'New Users'})
fig.update_layout(xaxis_tickangle=-45, height=450, hovermode='x unified')
fig.show()
save_fig(fig, '15_period_new_users')

# Author retention cohort: % of each month's new users still active N months later
cohort_rows = []
for _, row in new_p.iterrows():
    cohort_month = row['FirstMonth']
    cohort_authors = first_p[first_p['FirstMonth'] == cohort_month]['Author'].tolist()
    for lag in [1, 2, 3]:
        future = cohort_month + lag
        if future <= PERIOD_END:
            active = df_p[(df_p['Month'] == future) & (df_p['Author'].isin(cohort_authors))]['Author'].nunique()
            cohort_rows.append({
                'Cohort': str(cohort_month),
                'MonthsLater': lag,
                'RetentionPct': round(active / len(cohort_authors) * 100, 1) if cohort_authors else 0
            })

cohort_df = pd.DataFrame(cohort_rows)
fig2 = px.line(cohort_df, x='Cohort', y='RetentionPct', color='MonthsLater',
               title='New-User Retention — % Still Active N Months Later',
               labels={'Cohort': 'Cohort Month', 'RetentionPct': 'Retention %', 'MonthsLater': 'Months Later'},
               markers=True)
fig2.update_layout(xaxis_tickangle=-45, height=450, hovermode='x unified')
fig2.show()
save_fig(fig2, '15b_period_retention')


### 5.5 Message Length Distribution

In [ ]:
# Message length distribution (cap at 500 chars for readability)
msg_len = df_p['MsgLength'].clip(upper=500)

fig = px.histogram(msg_len, nbins=100,
                   title='Message Length Distribution — Apr 2023 to Jun 2024 (capped at 500 chars)',
                   labels={'value': 'Message Length (chars)', 'count': 'Frequency'})
fig.update_layout(height=400, showlegend=False)
fig.show()
save_fig(fig, '16_period_msg_length_dist')

# Median message length per month
med_len = (df_p.groupby('Month_str')['MsgLength']
           .median().reset_index(name='MedianLength'))
fig2 = px.line(med_len, x='Month_str', y='MedianLength', markers=True,
               title='Median Message Length per Month',
               labels={'Month_str': 'Month', 'MedianLength': 'Median Length (chars)'})
fig2.update_layout(xaxis_tickangle=-45, height=400, hovermode='x unified')
fig2.show()
save_fig(fig2, '16b_period_median_msg_length')


### 5.6 Suno Admin Activity in Period

In [ ]:
# Admin vs Community stacked bar for the period
df_p['IsAdmin'] = df_p['Author'].isin(SUNO_ADMIN)
split_p = (df_p.groupby(['Month_str', 'IsAdmin']).size()
           .unstack(fill_value=0)
           .rename(columns={False: 'Community', True: 'Admin'})
           .reset_index())

fig = go.Figure()
fig.add_bar(x=split_p['Month_str'], y=split_p['Community'], name='Community', marker_color='steelblue')
fig.add_bar(x=split_p['Month_str'], y=split_p['Admin'],     name='Suno Admin',  marker_color='tomato')
fig.update_layout(barmode='stack',
                  title='Monthly Messages: Admin vs Community — Apr 2023 to Jun 2024',
                  xaxis_tickangle=-45, height=450, hovermode='x unified')
fig.show()
save_fig(fig, '17_period_admin_vs_community')

# Admin share % per month
split_p['AdminPct'] = (split_p['Admin'] / (split_p['Admin'] + split_p['Community']) * 100).round(2)
fig2 = px.line(split_p, x='Month_str', y='AdminPct', markers=True,
               title='Suno Admin Message Share (%) per Month',
               labels={'Month_str': 'Month', 'AdminPct': 'Admin Share (%)'})
fig2.update_layout(xaxis_tickangle=-45, height=400, hovermode='x unified')
fig2.show()
save_fig(fig2, '17b_period_admin_share_pct')


### 5.7 Hour-of-Day Activity Pattern

In [ ]:
# Hour-of-day activity (UTC) for the period
hourly_p = df_p.groupby('Hour').size().reset_index(name='Messages')

fig = px.bar(hourly_p, x='Hour', y='Messages',
             title='Hourly Message Distribution (UTC) — Apr 2023 to Jun 2024',
             labels={'Hour': 'Hour of Day (UTC)', 'Messages': 'Total Messages'})
fig.update_layout(height=400, xaxis=dict(tickmode='linear', dtick=1))
fig.show()
save_fig(fig, '18_period_hourly_distribution')

# Day-of-week × Hour heatmap for the period
heatmap_p = df_p.groupby(['DayOfWeek', 'Hour']).size().reset_index(name='Messages')
pivot_ph = (heatmap_p
            .pivot(index='DayOfWeek', columns='Hour', values='Messages')
            .reindex(DOW_ORDER))

fig2 = px.imshow(pivot_ph,
                 title='Activity Heatmap: Day × Hour (UTC) — Apr 2023 to Jun 2024',
                 labels={'x': 'Hour (UTC)', 'y': 'Day of Week', 'color': 'Messages'},
                 color_continuous_scale='Blues', aspect='auto')
fig2.update_layout(height=400)
fig2.show()
save_fig(fig2, '18b_period_heatmap_dow_hour')
